# pl-review-sense — EDA & models

Polish review sentiment (3-class) on **PolEmo 2.0**: explore the data, train the **TF-IDF baseline**, compare with **HerBERT** (from the Colab run), and inspect errors.

Run from the repo root with the project venv. Outputs are intentionally not committed (PolEmo is CC BY-NC-SA — its text is not redistributed here).

In [ ]:
from collections import Counter
import matplotlib.pyplot as plt
from pl_review_sense import baseline, config, evaluate
from pl_review_sense.data import load_polemo, label_name

## Load PolEmo (3-class)

In [ ]:
ds = load_polemo()
print('train', len(ds.train), 'val', len(ds.validation), 'test', len(ds.test))

## Class distribution
Imbalanced — this is why macro-F1 is the headline metric.

In [ ]:
counts = Counter(ds.train.labels)
names = [label_name(i) for i in range(len(config.LABEL_NAMES))]
plt.bar(names, [counts[i] for i in range(len(names))])
plt.title('train class distribution'); plt.ylabel('reviews'); plt.show()

## Review length by class

In [ ]:
lengths = {i: [] for i in range(len(config.LABEL_NAMES))}
for t, y in zip(ds.train.texts, ds.train.labels):
    lengths[y].append(len(t.split()))
plt.boxplot([lengths[i] for i in range(len(names))], labels=names, showfliers=False)
plt.ylabel('words'); plt.title('review length (words) by class'); plt.show()

## Baseline: TF-IDF + logistic regression

In [ ]:
pipe = baseline.train(ds.train.texts, ds.train.labels)
y_pred = baseline.predict(pipe, ds.test.texts)
res = evaluate.evaluate(ds.test.labels, y_pred)
print(f'accuracy={res.accuracy:.4f}  macro_f1={res.macro_f1:.4f}')
for c in res.per_class:
    print(f'  {c.label:<9} P={c.precision:.3f} R={c.recall:.3f} F1={c.f1:.3f} n={c.support}')

## Confusion matrix (baseline)

In [ ]:
import numpy as np
cm = np.array(res.confusion)
fig, ax = plt.subplots()
ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(len(names)), labels=names); ax.set_yticks(range(len(names)), labels=names)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
for i in range(len(names)):
    for j in range(len(names)):
        ax.text(j, i, cm[i, j], ha='center', va='center')
plt.show()

## Compare with HerBERT (from the Colab run)
Populated once `reports/metrics/herbert.json` exists (see `herbert_colab.ipynb`).

In [ ]:
import json
p = config.HERBERT_METRICS_PATH
if p.exists():
    h = json.loads(p.read_text(encoding='utf-8'))
    print(f"baseline  macro_f1={res.macro_f1:.4f}")
    print(f"herbert   macro_f1={h['macro_f1']:.4f}  (run={h['run']})")
else:
    print('HerBERT metrics not found yet — run notebooks/herbert_colab.ipynb on a GPU.')

## Error analysis
Where does the baseline slip? Negation and sarcasm are usual suspects.

In [ ]:
errors = evaluate.misclassified(ds.test.texts, ds.test.labels, y_pred, limit=15)
print(f'{len(errors)} shown; negation share = {evaluate.negation_error_share(errors):.2f}')
for e in errors[:10]:
    print(f'[{e.true_label} -> {e.pred_label}] neg={e.negation} | {e.text[:140]}')

**Takeaway.** The TF-IDF baseline is already strong on this text-level task; HerBERT's gain (once run on GPU) should be weighed against its compute cost. Errors cluster around negation and contrastive/sarcastic phrasing — exactly where context helps.